In [1]:
import pandas as pd

## DATA GATHERING


In [2]:
data = pd.read_csv('huge_1M_titanic.csv')

In [3]:
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1310,1,1,"Name1310, Miss. Surname1310",female,NaN,0,0,SOTON/O2 3101272,76.760165,NaN,C
1,1311,0,3,"Name1311, Col. Surname1311",male,29.0,0,0,223596,10.193097,NaN,S
2,1312,0,3,"Name1312, Mr. Surname1312",male,20.0,0,0,54636,12.029416,C83,C
3,1313,0,3,"Name1313, Mr. Surname1313",male,27.0,0,0,PC 17760,13.429448,NaN,S
4,1314,0,3,"Name1314, Mr. Surname1314",male,32.0,0,0,364512,4.840769,E33,C


## DATA PREPROCESSING

In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 12 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   PassengerId  1000000 non-null  int64  
 1   Survived     1000000 non-null  int64  
 2   Pclass       1000000 non-null  int64  
 3   Name         1000000 non-null  str    
 4   Sex          1000000 non-null  str    
 5   Age          801400 non-null   float64
 6   SibSp        1000000 non-null  int64  
 7   Parch        1000000 non-null  int64  
 8   Ticket       1000000 non-null  str    
 9   Fare         1000000 non-null  float64
 10  Cabin        229805 non-null   str    
 11  Embarked     997760 non-null   str    
dtypes: float64(2), int64(5), str(5)
memory usage: 132.6 MB


In [5]:
data.isnull().sum()

PassengerId         0
Survived            0
Pclass              0
Name                0
Sex                 0
Age            198600
SibSp               0
Parch               0
Ticket              0
Fare                0
Cabin          770195
Embarked         2240
dtype: int64

In [6]:
data = data.drop(columns = ['PassengerId','Name','Age','Ticket','Cabin'])

In [7]:
data.head()

,Survived,Pclass,Sex,SibSp,Parch,Fare,Embarked
0,1,1,female,0,0,76.760165,C
1,0,3,male,0,0,10.193097,S
2,0,3,male,0,0,12.029416,C
3,0,3,male,0,0,13.429448,S
4,0,3,male,0,0,4.840769,C


## DATA CLEANING

In [8]:
data['Embarked'] = data['Embarked'].replace({'S':'Southampton','C':'Chebourg','Q':'Queenstown'})

In [9]:
data.head()

,Survived,Pclass,Sex,SibSp,Parch,Fare,Embarked
0,1,1,female,0,0,76.760165,Chebourg
1,0,3,male,0,0,10.193097,Southampton
2,0,3,male,0,0,12.029416,Chebourg
3,0,3,male,0,0,13.429448,Southampton
4,0,3,male,0,0,4.840769,Chebourg


In [10]:
data.dropna(subset=['Embarked'],inplace = True)

In [11]:
data.isnull().sum()

Survived    0
Pclass      0
Sex         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

## FEATURE ENCODING

In [12]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

In [13]:
data['Fare'] = data['Fare'].astype('int')

In [14]:
label = LabelEncoder()
onehot = OneHotEncoder(sparse_output = False)

In [15]:
data['Sex'] = label.fit_transform(data['Sex'])

In [16]:
Embarked = onehot.fit_transform(data[['Embarked']])

In [17]:
Embarked = pd.DataFrame(Embarked, columns = onehot.get_feature_names_out())

In [18]:
data = pd.concat([data.drop(columns = ['Embarked']),Embarked], axis=1)

## FEATURE SCALING

In [19]:
from sklearn.preprocessing import StandardScaler

In [20]:
scale = StandardScaler()

In [21]:
num_cols = ['Pclass', 'SibSp', 'Parch', 'Fare']

In [22]:
data[num_cols] = scale.fit_transform(data[num_cols])

In [23]:
data.head()

,Survived,Pclass,Sex,SibSp,Parch,Fare,Embarked_Chebourg,Embarked_Queenstown,Embarked_Southampton
0,1.0,-1.568865,0.0,-0.462644,-0.469267,0.896610,1.0,0.0,0.0
1,0.0,0.824107,1.0,-0.462644,-0.469267,-0.479488,0.0,0.0,1.0
2,0.0,0.824107,1.0,-0.462644,-0.469267,-0.437788,1.0,0.0,0.0
3,0.0,0.824107,1.0,-0.462644,-0.469267,-0.416938,0.0,0.0,1.0
4,0.0,0.824107,1.0,-0.462644,-0.469267,-0.604587,1.0,0.0,0.0


In [24]:
import pickle

In [25]:
with open('label_encoder.pkl', 'wb') as file:
    pickle.dump(label,file)

with open('onehot_encoder.pkl', 'wb') as file:
    pickle.dump(onehot,file)

with open('scalar_encoder.pkl', 'wb') as file:
    pickle.dump(scale,file)

# Train - Validation - Test SPLIT

In [26]:
from sklearn.model_selection import train_test_split

In [27]:
data = data.dropna()

In [28]:
X = data.drop(columns = ['Survived'])
y = data['Survived']

In [29]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 20, random_state = 42)

In [30]:
X_train,X_valid,y_train,y_valid = train_test_split(X_train,y_train, test_size = 0.2, random_state = 42)

## Model Building


In [31]:
import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

In [32]:
model = Sequential([Dense(64,input_shape = (X_train.shape[1],),activation = 'relu'), #First hidden Layer
            Dense(32,activation = 'relu'), #Second hidden Layer
            Dense(1,activation = 'sigmoid')]) #Output Layer

d:\Praticals\Generative AI\Deep Learning\TitanicProject\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [33]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,689 (10.50 KB)

 Trainable params: 2,689 (10.50 KB)

 Non-trainable params: 0 (0.00 B)

## Model Compilation
#### Optimizer?
#### Loss?
#### Metrics?

In [34]:
opt = tensorflow.keras.optimizers.Adam(learning_rate = 0.01)
tensorflow.keras.losses.BinaryFocalCrossentropy

keras.src.losses.losses.BinaryFocalCrossentropy

In [35]:
model.compile(optimizer = opt , loss = 'binary_crossentropy', metrics = ['accuracy'])

In [36]:
stopping_callback = EarlyStopping(monitor = 'val_loss', patience = 5, restore_best_weights = True)

In [37]:
model.fit(X_train,y_train,validation_data = (X_valid,y_valid), epochs = 100, callbacks = [stopping_callback])

Epoch 1/100
24888/24888 ━━━━━━━━━━━━━━━━━━━━ 34s 1ms/step - accuracy: 0.8466 - loss: 0.3478 - val_accuracy: 0.8495 - val_loss: 0.3431
Epoch 2/100
24888/24888 ━━━━━━━━━━━━━━━━━━━━ 36s 1ms/step - accuracy: 0.8517 - loss: 0.3348 - val_accuracy: 0.8556 - val_loss: 0.3275
Epoch 3/100
24888/24888 ━━━━━━━━━━━━━━━━━━━━ 36s 1ms/step - accuracy: 0.8525 - loss: 0.3331 - val_accuracy: 0.8524 - val_loss: 0.3272
Epoch 4/100
24888/24888 ━━━━━━━━━━━━━━━━━━━━ 35s 1ms/step - accuracy: 0.8522 - loss: 0.3327 - val_accuracy: 0.8560 - val_loss: 0.3295
Epoch 5/100
24888/24888 ━━━━━━━━━━━━━━━━━━━━ 40s 2ms/step - accuracy: 0.8527 - loss: 0.3319 - val_accuracy: 0.8549 - val_loss: 0.3258
Epoch 6/100
24888/24888 ━━━━━━━━━━━━━━━━━━━━ 38s 2ms/step - accuracy: 0.8527 - loss: 0.3319 - val_accuracy: 0.8550 - val_loss: 0.3280
Epoch 7/100
24888/24888 ━━━━━━━━━━━━━━━━━━━━ 37s 1ms/step - accuracy: 0.8531 - loss: 0.3315 - val_accuracy: 0.8553 - val_loss: 0.3314
Epoch 8/100
24888/24888 ━━━━━━━━━━━━━━━━━━━━ 37s 1ms/step - ac

In [38]:
model.save('model.h5')